# Лабораторна. PokéData: аналіз персонажів через API

**Тривалість:** 60 хвилин

**Контекст.** Ви — аналітик студії, яка робить мобільну гру-колекціонування в стилі Pokémon. Геймдизайнери хочуть зрозуміти баланс оригінальної гри: які типи персонажів найпоширеніші, які найсильніші і чи пов'язана сила персонажа з його вагою. Дані про всіх персонажів є у відкритому API.

**Сервіс:** [PokeAPI](https://pokeapi.co) — безкоштовний API з даними про покемонів, який не вимагає реєстрації чи ключа. Ми працюватимемо з першими 151 покемонами (перше покоління гри).

Порівняно з лабораторною WeatherWise тут є три нові речі:
1. список персонажів віддається **частинами** (пагінація), і треба пройти всі частини;
2. дані кожного персонажа лежать за **окремою адресою**, тож запитів буде багато, і результати варто **кешувати**;
3. відповіді дуже **вкладені** (списки словників у словниках), і їх треба перетворити на просту таблицю.

**План роботи:**

| Частина | Що робимо | Час |
|---|---|---|
| 0 | Підготовка | 2 хв |
| 1 | Список персонажів і пагінація | 12 хв |
| 2 | Дані одного персонажа | 15 хв |
| 3 | Збираємо всіх персонажів з кешем | 10 хв |
| 4 | Аналіз | 14 хв |
| 5 | Збереження в CSV | 4 хв |
| 6 | Підсумок | 3 хв |

**Що здати:** цей ноутбук із заповненим кодом і відповідями в комірках «Ваша відповідь», а також файли `pokemon_cache.json` і `pokemon.csv`.

**Оцінювання:** максимум 1.25 бала. Бали вказано біля кожного завдання. Якщо кілька завдань оцінюються разом, бали ставляться, лише коли виконано всі з них. Код, що падає з помилкою, і відповіді без пояснення не зараховуються.

## Частина 0. Підготовка (2 хв)

Крім знайомих модулів, нам знадобиться `os` — вбудований модуль для роботи з файловою системою (перевіримо, чи існує файл кешу).

Функція `get_json(url, params=None)` уже готова: вона робить запит, перевіряє код відповіді через `raise_for_status()` і повертає результат `response.json()`. Так ми не повторюватимемо ці рядки в кожному завданні.

In [ ]:
# !pip install requests

import csv
import json
import os
import time

import requests

API_URL = "https://pokeapi.co/api/v2"

def get_json(url, params=None):
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()

print("Усе готово!")

## Частина 1. Список персонажів і пагінація (12 хв)

Endpoint `https://pokeapi.co/api/v2/pokemon` повертає список персонажів. Їх понад тисячу, тому API віддає їх **частинами** (сторінками). Розмір частини й місце, з якого почати, задаються параметрами:
- `limit` — скільки елементів повернути за раз;
- `offset` — скільки елементів пропустити від початку.

Тобто `limit=20, offset=0` — перші 20 персонажів, `limit=20, offset=20` — з 21-го по 40-й, і так далі.

**Завдання 1.1.** *(0.25 б. за 1.1–1.3)* Зробіть запит до `f"{API_URL}/pokemon"` з параметрами `limit=5` і `offset=0`. Виведіть ключі відповіді та значення кожного ключа. Дайте відповідь:
1. Скільки всього персонажів в API?
2. Що лежить у ключі `results`? Що є в кожному елементі?
3. Що лежить у ключах `next` і `previous`? Чому `previous` дорівнює `None`?

In [ ]:
# TODO

**Ваша відповідь:** 

### Пагінація через `next`

API саме підказує, де взяти наступну частину: у ключі `next` лежить **готова адреса** з потрібними `limit` і `offset`. Тому найпростіший спосіб пройти всі частини — цикл `while`, який переходить за `next`, доки він не `None`. Це той самий підхід, що й з кнопкою «Next» на сайтах, тільки тут кнопка — це ключ у JSON.

Нам потрібні не всі персонажі, а лише перші 151, тож зупинимося, щойно набереться потрібна кількість.

**Завдання 1.2.** *(0.25 б. за 1.1–1.3)* Напишіть функцію `get_pokemon_list(total=151, page_size=50)`, яка повертає список словників `{"name": ..., "url": ...}` для перших `total` персонажів.

Кроки:
1. Створіть порожній список `pokemon` і змінну `url` з адресою першої частини: ``f"{API_URL}/pokemon?limit={page_size}&offset=0"``.
2. Напишіть цикл `while`, який працює, доки `url` не `None` **і** в списку менше `total` елементів.
3. Усередині циклу завантажте частину через `get_json(url)`, додайте її `results` до списку через `.extend()`, а в `url` запишіть значення `next`.
4. Зробіть паузу `time.sleep(0.2)`.
5. Після циклу в списку може бути більше, ніж `total` (остання частина — 50 елементів). Поверніть лише перші `total` за допомогою зрізу.

Скільки частин (запитів) знадобилося, щоб отримати 151 персонажа?

In [ ]:
def get_pokemon_list(total=151, page_size=50):
    # TODO
    ...

pokemon_list = get_pokemon_list()
print(len(pokemon_list))
print(pokemon_list[0], pokemon_list[-1])

**Ваша відповідь:** 

**Питання 1.3.** *(0.25 б. за 1.1–1.3)* Можна було б не переходити за `next`, а самостійно рахувати `offset` (0, 50, 100, ...). Чому варіант з `next` надійніший? Підказка: подумайте, що станеться, якщо розробники API змінять спосіб пагінації (наприклад, з `offset` на номер сторінки `page`).

**Ваша відповідь:** 

## Частина 2. Дані одного персонажа (15 хв)

Повні дані персонажа лежать за адресою з ключа `url` (або `f"{API_URL}/pokemon/{name}"`). Відповідь дуже велика: сотні рядків з атаками, спрайтами, версіями гри. Нам потрібна лише невелика частина.

**Завдання 2.1.** *(0.25 б. за 2.1–2.3)* Завантажте дані персонажа `pikachu` і дослідіть структуру відповіді:
1. Виведіть усі ключі відповіді.
2. Виведіть значення ключів `id`, `name`, `height`, `weight`.
3. Виведіть **повністю** значення ключів `types` і `stats` і роздивіться їх: що всередині кожного елемента списку?

In [ ]:
# TODO

### Розбираємо вкладені дані

Типи й характеристики — це **списки словників, у яких теж є словники**. Наприклад, `types` виглядає так:
```python
[{"slot": 1, "type": {"name": "electric", "url": "..."}}]
```
Щоб отримати назву типу, треба пройти два рівні: елемент списку → ключ `type` → ключ `name`. Для всіх типів одразу — list comprehension:
```python
[t["type"]["name"] for t in pikachu["types"]]   # ['electric']
```

`stats` — список з 6 характеристик, і в кожній є значення `base_stat` і назва `stat` → `name`:
```python
[{"base_stat": 35, "effort": 0, "stat": {"name": "hp", "url": "..."}}, ...]
```
Зручніше перетворити його на словник «назва → значення» через **dict comprehension** — це те саме, що list comprehension, але у фігурних дужках і з парою `ключ: значення`:
```python
{s["stat"]["name"]: s["base_stat"] for s in pikachu["stats"]}
# {'hp': 35, 'attack': 55, 'defense': 40, 'special-attack': 50, 'special-defense': 50, 'speed': 90}
```

**Одиниці виміру.** API зберігає зріст у **дециметрах**, а вагу — в **гектограмах** (100 г). Щоб отримати метри й кілограми, обидва значення треба поділити на 10.

**Завдання 2.2.** *(0.25 б. за 2.1–2.3)* Напишіть функцію `parse_pokemon(data)`, яка приймає повну відповідь API про персонажа й повертає «плаский» словник:

| ключ | тип | приклад для pikachu |
|---|---|---|
| `id` | int | `25` |
| `name` | str | `pikachu` |
| `types` | list[str] | `["electric"]` |
| `height_m` | float | `0.4` |
| `weight_kg` | float | `6.0` |
| `hp`, `attack`, `defense`, `speed` | int | `35`, `55`, `40`, `90` |
| `total_stats` | int | сума всіх 6 характеристик: `320` |

Кроки:
1. Зберіть список назв типів (list comprehension).
2. Зберіть словник характеристик (dict comprehension).
3. Переведіть зріст і вагу в метри й кілограми.
4. Поверніть словник; значення `hp`, `attack`, `defense`, `speed` візьміть зі словника характеристик, а `total_stats` — сума **всіх** його значень (`sum(stats.values())`).

In [ ]:
def parse_pokemon(data):
    # TODO
    ...

print(parse_pokemon(pikachu))

**Завдання 2.3.** *(0.25 б. за 2.1–2.3)* Що повертає API, якщо персонажа не існує? Напишіть функцію `get_pokemon(name)`, яка повертає результат `parse_pokemon` або `None`, якщо персонажа з таким ім'ям немає.

Кроки:
1. Зробіть запит через `requests.get` (не через `get_json`, бо `raise_for_status()` зупинить програму).
2. Якщо `response.status_code` дорівнює 404, поверніть `None`.
3. Інакше викличте `response.raise_for_status()` (на випадок інших помилок) і поверніть `parse_pokemon(response.json())`.

Перевірте на `charmander` і на `pikachuu` (з помилкою). Який код відповіді API повертає для неіснуючого персонажа?

In [ ]:
def get_pokemon(name):
    # TODO
    ...

print(get_pokemon("charmander"))
print(get_pokemon("pikachuu"))   # None

## Частина 3. Збираємо всіх персонажів з кешем (10 хв)

Щоб отримати дані 151 персонажа, потрібен 151 запит. З паузою це займе близько хвилини. Якщо ноутбук перезапустити, усі запити доведеться повторити, хоча дані про персонажів не змінюються.

**Рішення — кеш у файлі.** Після першого збору зберігаємо результат у JSON-файл. Під час наступного запуску спершу перевіряємо, чи файл уже є: якщо так — читаємо дані з нього й не робимо жодного запиту. Перевірити, чи існує файл, можна функцією `os.path.exists("назва_файлу")` — вона повертає `True` або `False`.

Нагадування з лабораторної WeatherWise:
```python
json.dump(дані, f, ensure_ascii=False, indent=2)   # запис у відкритий файл f
дані = json.load(f)                                # читання з відкритого файлу f
```

**Завдання 3.1.** *(0.25 б. за 3.1 + 3.2)* Напишіть функцію `load_all_pokemon(pokemon_list, cache_file="pokemon_cache.json")`, яка повертає список «пласких» словників для всіх персонажів зі списку.

Кроки:
1. Якщо файл кешу існує, відкрийте його, прочитайте список через `json.load`, виведіть повідомлення «Дані з кешу» і поверніть список.
2. Інакше створіть порожній список і пройдіться по `pokemon_list`. Для кожного елемента завантажте повні дані за його `url` через `get_json`, розберіть через `parse_pokemon` і додайте в список.
3. Кожні 25 персонажів виводьте прогрес (підказка: `if len(список) % 25 == 0:`). Після кожного запиту робіть паузу `time.sleep(0.2)`.
4. Після циклу збережіть список у файл кешу через `json.dump` і поверніть його.

Запустіть функцію **двічі** і порівняйте, скільки часу зайняв кожен запуск.

In [ ]:
def load_all_pokemon(pokemon_list, cache_file="pokemon_cache.json"):
    # TODO
    ...

all_pokemon = load_all_pokemon(pokemon_list)
print(len(all_pokemon))

In [ ]:
# Другий запуск: має бути миттєвим
all_pokemon = load_all_pokemon(pokemon_list)
print(len(all_pokemon))

**Питання 3.2.** *(0.25 б. за 3.1 + 3.2)* Скільки часу зайняв перший і другий запуск? У якій ситуації кеш стає **проблемою**? (Підказка: що буде, якщо ви виправите помилку в `parse_pokemon` і запустите функцію знову?) Як цю проблему вирішити?

**Ваша відповідь:** 

## Частина 4. Аналіз (14 хв)

Аналізуємо список `all_pokemon` на чистому Python. Нагадування прийомів:

| Що хочемо | Код |
|---|---|
| Підрахунок у словнику | `counts[ключ] = counts.get(ключ, 0) + 1` |
| Групування у словник зі списками | `groups.setdefault(ключ, []).append(значення)` |
| Елемент з найбільшим значенням | `max(список, key=функція)` |
| Сортування ключів словника за значеннями | `sorted(словник, key=словник.get, reverse=True)` |
| Фільтр | `[x for x in список if умова]` |

**Завдання 4.1.** *(0.25 б. за 4.1–4.3)* Які типи найпоширеніші? Порахуйте, скільки персонажів має кожен тип, і виведіть топ-5.

**Увага:** у персонажа може бути **два** типи (наприклад, `["grass", "poison"]`), і рахувати треба обидва. Тому знадобиться **вкладений цикл**: зовнішній — по персонажах, внутрішній — по типах персонажа.

In [ ]:
# TODO

**Ваша відповідь:** 

**Завдання 4.2.** *(0.25 б. за 4.1–4.3)* Скільки персонажів мають два типи, а скільки — один? Яку частку (у відсотках) становлять персонажі з двома типами?

In [ ]:
# TODO

**Завдання 4.3.** *(0.25 б. за 4.1–4.3)* Які типи найсильніші? Для кожного типу порахуйте середнє `total_stats` його персонажів (знову з урахуванням обох типів) і виведіть усі типи від найсильнішого до найслабшого разом з кількістю персонажів.

Кроки:
1. Згрупуйте `total_stats` за типами у словник зі списками (вкладений цикл, як у 4.1).
2. Створіть словник «тип → середнє», округлене до 1 знака.
3. Відсортуйте типи за середнім від більшого до меншого й виведіть.

Чи можна довіряти середньому для типу, у якого лише 1–3 персонажі?

In [ ]:
# TODO

**Ваша відповідь:** 

**Завдання 4.4.** *(0.25 б. за 4.4 + 5.1 + 6.1–6.2)* Геймдизайнери питають: **чи важчі персонажі сильніші?** Розділіть персонажів на три вагові категорії й порахуйте для кожної кількість персонажів і середнє `total_stats`:
- «легкі» — до 10 кг;
- «середні» — від 10 до 100 кг;
- «важкі» — від 100 кг.

Кроки:
1. Напишіть функцію `weight_class(kg)`, яка повертає назву категорії за вагою (`if` / `elif` / `else`).
2. Згрупуйте `total_stats` за категоріями.
3. Виведіть для кожної категорії кількість і середнє.

Також виведіть найважчого й найлегшого персонажа (ім'я та вагу). Чи підтверджують дані гіпотезу геймдизайнерів?

In [ ]:
# TODO

**Ваша відповідь:** 

## Частина 5. Збереження в CSV (4 хв)

Наш «плаский» словник майже готовий для таблиці, але в ключі `types` лежить **список**, а в комірці CSV може бути лише одне значення. Якщо записати список як є, у файлі з'явиться текст на кшталт `['grass', 'poison']`, з яким незручно працювати в Excel.

Типовий спосіб — перетворити список на рядок через роздільник, наприклад `"/".join(["grass", "poison"])` → `"grass/poison"`. Метод `join` склеює елементи списку рядків, вставляючи між ними рядок, на якому його викликали.

**Завдання 5.1.** *(0.25 б. за 4.4 + 5.1 + 6.1–6.2)* Збережіть усіх персонажів у файл `pokemon.csv` з колонками `id`, `name`, `types`, `height_m`, `weight_kg`, `hp`, `attack`, `defense`, `speed`, `total_stats`. У колонці `types` типи мають бути записані через `/`.

Кроки:
1. Створіть новий список словників: для кожного персонажа зробіть **копію** словника (`dict(p)`) і замініть у копії `types` на рядок через `join`. Копія потрібна, щоб не зіпсувати оригінальний список `all_pokemon`.
2. Збережіть новий список через `csv.DictWriter`.
3. Відкрийте файл і перевірте, що в ньому 152 рядки (заголовок і 151 персонаж).

In [ ]:
# TODO

## Частина 6. Підсумок (3 хв)

**6.1.** *(0.25 б. за 4.4 + 5.1 + 6.1–6.2)* У цій лабораторній ми зберегли дані у **двох** форматах: JSON (кеш) і CSV. Чому для кешу краще підходить JSON, а для передачі геймдизайнерам — CSV?

**6.2.** *(0.25 б. за 4.4 + 5.1 + 6.1–6.2)* Ми зробили 4 запити для списку і 151 запит для деталей. Чи можна було обійтися меншою кількістю запитів, якби API був влаштований інакше? Що б ви запропонували розробникам API?

**Ваша відповідь:** 

## Бонус (якщо залишився час) · не оцінюється

**Б.1.** Endpoint `f"{API_URL}/type/{назва}"` повертає інформацію про тип, зокрема в ключі `damage_relations` → `double_damage_to` — список типів, проти яких цей тип завдає подвійної шкоди. Для 5 найпоширеніших типів з 4.1 виведіть, проти кого вони найефективніші.

**Б.2.** Знайдіть персонажа з найбільшим `total_stats` серед тих, хто легший за 10 кг. Це «найкращий кишеньковий боєць» для геймдизайнерів.

In [ ]:
# Бонус